In [1]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [2]:
from openai import OpenAI

openai_client = OpenAI()

In [3]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure it out quickly. Send me:\n- the course name\n- the institution/platform\n- when it starts or started\n\nAnd I can help you draft a message to the instructor or registrar, for example:\n\n> Hi, I just discovered this course and I’m very interested in joining. Is it still possible to enroll late? If so, please let me know the next steps. Thank you.\n\nIf you’d like, I can also make this message more formal or more casual.'

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system.\n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux: run:\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. Open a terminal and run:\n```bash\nollama run llama3\n```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. If you want to test the local server, run:\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```'

In [6]:
assistant.rag("How do I run Olama locally?")

'The context doesn’t mention **Olama/Ollama** specifically.\n\nIf you meant running the course **locally**, the FAQ says you can do that instead of Codespaces if you’re comfortable setting up the needed tools:\n\n- Python\n- `uv`\n- Jupyter\n- Docker\n- any other module-specific tools\n\nIt also says that if you run locally, you should:\n\n- document your setup\n- keep your environment reproducible\n\nIf you meant something else by “Olama,” let me know and I can help based on the FAQ context.'

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

call=response.output[0]

In [10]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join FAQ"}', call_id='call_XJGVwOop8gcWP2225BX8lCFx', name='search', type='function_call', id='fc_07a6348aaeee18c5006a3c7d42de8081a1914e21135a2a13a3', namespace=None, status='completed')

In [11]:
import json

args=json.loads(call.arguments)

In [12]:
results = search(**args)

In [13]:
call.call_id

'call_XJGVwOop8gcWP2225BX8lCFx'

In [14]:
results_json = json.dumps(results, indent=2)

In [15]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": results_json
}

In [16]:
messages.append(call)

In [17]:
messages.append(function_call_output)

In [18]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join FAQ"}', call_id='call_XJGVwOop8gcWP2225BX8lCFx', name='search', type='function_call', id='fc_07a6348aaeee18c5006a3c7d42de8081a1914e21135a2a13a3', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_XJGVwOop8gcWP2225BX8lCFx',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to

In [19]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

In [21]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open.


In [22]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()